In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer
import joblib

In [ ]:
# Load data

url='https://raw.githubusercontent.com/louiee-jason/bank-customer-segmentation-and-classification/main/data/bank_transactions_data_edited.csv'
df = pd.read_csv(url)


In [ ]:
# Tampilkan 5 baris pertama dengan function head.
df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68.0,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19.0,Student,56.0,1.0,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26.0,Student,25.0,1.0,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,NaN,Student,198.0,1.0,7429.40,2024-11-04 08:06:39


In [ ]:
# Tinjau jumlah baris kolom dan jenis data dalam dataset dengan info.
df.info()

In [ ]:
# Menampilkan statistik deskriptif dataset dengan menjalankan describe
df.describe()

In [ ]:
# Menampilkan korelasi antar fitur
# Memilih kolom numerik
numerical_cols = df.select_dtypes(include=['number']).columns

# Hitung matriks korelasi
correlation = df[numerical_cols].corr()

# Buat visualisasi heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(correlation,
               annot=True,
               cmap='coolwarm',
               fmt=".2f",
               vmin=-1,
               vmax=1)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Menampilkan histogram untuk semua kolom numerik (Opsional Skilled 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, column in enumerate(numerical_cols):

    # Tampilkan histogram dan pastikan plot ditempatkan di subplot (axes) yang benar
    sns.histplot(df[column], bins=20, kde=True, color='skyblue', ax=axes[i])

    # Atur judul dan label
    axes[i].set_title(column)
    axes[i].set_xlabel("Nilai")
    axes[i].set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi yang lebih informatif
plt.figure(figsize=(12, 6))

# Buat visualisasi boxplot untuk melihat sebaran 'TransactionAmount' (y) berdasarkan 'CustomerOccupation' (x)
sns.boxplot(x="CustomerOccupation", y="TransactionAmount", data=df)

plt.title("Nilai Transaksi per Pekerjaan Nasabah (Boxplot)")

# Putar label sumbu-x agar tidak tumpang tindih
plt.xticks(rotation=45)

plt.show()

#Visualisasi Violin Plot
sns.violinplot(x="CustomerOccupation", y="TransactionAmount", data=df)
plt.title("Nilai Transaksi per Pekerjaan Nasabah (Violinplot)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Mengecek dataset menggunakan isnull().sum()
df.isnull().sum()

In [ ]:
# Mengecek dataset menggunakan duplicated().sum()
df.duplicated().sum()

In [ ]:
# Handling missing values
df.dropna(inplace=True)
# Cek kembali dataset menggunakan isnull().sum()
df.isnull().sum()

In [ ]:
# Handling duplicates values
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
# Melakukan drop pada kolom yang memiliki keterangan unique
cols_to_drop = [col for col in df if
                'id' in col.lower() or
                'ip' in col.lower() or
                'date' in col.lower()]
df = df.drop(columns=cols_to_drop)
df.head()

In [ ]:
# Melakukan feature encoding menggunakan LabelEncoder() untuk fitur kategorikal.
categorical_cols = list(df.select_dtypes(include=['object']).columns)

encoders = {}

# Loop melalui setiap kolom kategorikal
for column in categorical_cols:
    # Buat (instantiate) objek LabelEncoder
    label_encoder = LabelEncoder()

    # Terapkan (fit) encoder ke data dan sekaligus ubah (transform) data tersebut
    df[column] = label_encoder.fit_transform(df[column])

    # Simpan encoder
    encoders[column] = label_encoder
df.head()

In [ ]:
# Last checking gunakan columns.tolist() untuk checking seluruh fitur yang ada.
df.columns.tolist()

In [ ]:
# Melakukan Handling Outlier Data menggunakan metode drop.

for col in numerical_cols:
    # Hitung Kuartil 1 (Q1) dan Kuartil 3 (Q3)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # Hitung Interquartile Range (IQR)
    IQR = Q3 - Q1

    # Tentukan batas bawah (lower bound) dan batas atas (upper bound)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter DataFrame: Simpan hanya baris di mana nilai 'df[col]' berada DI ANTARA (inklusif) batas bawah dan batas atas.
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

# Tampilkan statistik deskriptif setelah outlier dihapus
df.describe()

In [ ]:
# Melakukan feature scaling menggunakan StandardScaler() untuk fitur numerik.
# Buat (instantiate) StandardScaler
scaler = StandardScaler()
# Terapkan (fit) scaler ke data dan sekaligus ubah (transform) data tersebut
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Melakukan binning data berdasarkan kondisi rentang nilai pada fitur numerik

#1. Customer Age
col_to_bin = 'CustomerAge'
new_col_name = 'CustomerAge_bin'
bin_labels = ['Muda', 'Dewasa', 'Tua']
df[new_col_name] = pd.qcut(df[col_to_bin], q=3, labels=bin_labels, duplicates='drop')

label_encoder = LabelEncoder()
df[new_col_name] = label_encoder.fit_transform(df[new_col_name])

encoders[new_col_name] = label_encoder
categorical_cols.extend([new_col_name])

#2. TransactionAmount
col_to_bin = 'TransactionAmount'
new_col_name = 'TransactionAmount_bin'
bin_labels = ['Kecil', 'Sedang', 'Besar']
df[new_col_name] = pd.qcut(df[col_to_bin], q=3, labels=bin_labels, duplicates='drop')
label_encoder = LabelEncoder()
df[new_col_name] = label_encoder.fit_transform(df[new_col_name])
encoders[new_col_name] = label_encoder
categorical_cols.extend([new_col_name])

df.head()

In [ ]:
# Gunakan describe untuk memastikan proses clustering menggunakan dataset hasil preprocessing

# Buat salinan (copy) dari 'df' ke variabel 'df_used'
df_used = df.copy()
# Tampilkan ringkasan statistik dari DataFrame 'df'
df_used.describe()

In [ ]:
# Melakukan visualisasi Elbow Method menggunakan KElbowVisualizer()

model = KMeans()
visualizer = KElbowVisualizer(model,
                       k=(2,10),
                       metric='silhouette',
                       timings=False)
visualizer.fit(df)
visualizer.show()

In [ ]:
# Menggunakan algoritma K-Means Clustering
model = KMeans(n_clusters=2, random_state=42)
model.fit(df)

In [ ]:
# Menyimpan model menggunakan joblib
joblib.dump(model, "model_clustering.h5")

In [ ]:
# Menghitung dan menampilkan nilai Silhouette Score.
labels = model.labels_
score = silhouette_score(df, labels)

print("Silhouette Score:", score)

In [ ]:
# Membuat visualisasi hasil clustering

# Buat (instantiate) objek PCA untuk 2 komponen (n_components=2)
pca = PCA(n_components=2)

# Terapkan (fit) PCA ke data 'df' dan transformasikan data tersebut
df_pca = pca.fit_transform(df)

# Buat DataFrame baru 'df_pca' dari hasil transformasi
df_pca = pd.DataFrame(data=df_pca, columns=['Principal Component 1', 'Principal Component 2'])
df_pca['Cluster'] = labels

# Buat scatter plot menggunakan Seaborn
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='Principal Component 1',
    y='Principal Component 2',
    hue='Cluster',  # Warnai titik berdasarkan kolom 'Cluster'
    palette=sns.color_palette("viridis", n_colors=2),
    data=df_pca,
    legend="full",
    alpha=0.8
)

plt.title('Visualisasi Cluster dalam 2D (menggunakan PCA)', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
centers = pca.transform(model.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.7, marker='X', label='Centroid')
plt.legend()
plt.show()

In [ ]:
# Membangun model menggunakan PCA.
pca = PCA(n_components=2)

# Terapkan (fit) PCA ke data 'df_used' dan transformasikan data tersebut
df_pca_array = pca.fit_transform(df_used)

# Buat DataFrame baru 'data_final' dari hasil array PCA
data_final = pd.DataFrame(data=df_pca_array, columns=['PCA1', 'PCA2'])
kmeans_pca = KMeans(n_clusters=2, random_state=42)
kmeans_pca.fit(data_final)

In [ ]:
joblib.dump(kmeans_pca, "PCA_model_clustering.h5")

## **a. Interpretasi Hasil Clustering**


In [ ]:
# Menampilkan analisis deskriptif minimal mean, min dan max untuk fitur numerik.

df_used['Cluster'] = labels
agg_summary = df_used.groupby('Cluster')[numerical_cols].agg(['mean', 'min', 'max']).round(2).T

# Tampilkan hasil ringkasan
display(agg_summary)

## Menjelaskan karakteristik tiap cluster berdasarkan rentangnya sebelum **Inverse**
1. **CLUSTER 1: ([Casual & Low Engagement Users])**:
  - **Rata-rata (mean) TransactionAmount:** <Fitur>: -0.01
  - **Rata-rata (mean) CustomerAge:** <Fitur>: 0.02
  - **Rata-rata (mean) TransactionDuration:** <Fitur>: 0.03
  - **Rata-rata (mean) LoginAttempts:** <Fitur>: 0.00
  - **Rata-rata (mean) AccountBalance:** <Fitur>: 0.01
  <Sebelum inverse>
  - **Analisis:** Cluster ini menggambarkan kelompok pengguna dengan usia yang relatif lebih muda, durasi transaksi yang lebih singkat, serta saldo akun yang lebih rendah, sehingga mencerminkan tingkat aktivitas yang tidak terlalu intens. Pengguna dalam cluster ini cenderung menggunakan sistem secara lebih sederhana dan tidak terlalu sering, sehingga dapat dikategorikan sebagai pengguna dengan keterlibatan yang lebih rendah. Sehingga rekomendasi pada kelompok nasabah ini adalah dengan memberikan promosi, cashback, atau program loyalitas untuk meningkatkan keterlibatan, serta menawarkan produk yang sederhana dan mudah digunakan agar nasabah lebih aktif dalam bertransaksi.

2. **CLUSTER 0: Established & Engaged Users**:
  - **Rata-rata (mean) TransactionAmount:** <Fitur>: 0.01
  - **Rata-rata (mean) CustomerAge:** <Fitur>: -0.02
  - **Rata-rata (mean) TransactionDuration:** <Fitur>: -0.03
  - **Rata-rata (mean) LoginAttempts:** <Fitur>: 0.00
  - **Rata-rata (mean) AccountBalance:** <Fitur>: -0.01

  - **Analisis:** Cluster ini menggambarkan kelompok pengguna dengan usia yang relatif lebih matang, durasi transaksi yang lebih lama, serta saldo akun yang lebih tinggi, sehingga menunjukkan pola penggunaan yang lebih stabil. Pengguna dalam cluster ini cenderung memiliki aktivitas yang lebih konsisten dan dapat dikategorikan sebagai pengguna yang lebih berpengalaman serta memiliki keterlibatan yang lebih tinggi dalam sistem. Sehingga rekomendasi pada kelompok nasabah ini adalah dengan menawarkan produk premium seperti investasi, tabungan berjangka, atau layanan prioritas, karena mereka cenderung memiliki keterlibatan dan kemampuan finansial yang lebih baik.

In [ ]:
df_used.rename(columns={"Cluster": "Target"}, inplace=True)

df_used.head()

In [ ]:
# Simpan Data
df_used.to_csv('data_clustering.csv', index=False)

In [ ]:
# inverse dataset ke rentang normal untuk numerikal

df_inverse = df_used.copy()
df_inverse[numerical_cols] = scaler.inverse_transform(df_inverse[numerical_cols])

df_inverse.head()

In [ ]:
# inverse dataset yang sudah diencode ke kategori aslinya.

for column in categorical_cols:
    encoder = encoders[column]
    df_inverse[column] = encoder.inverse_transform(df_inverse[column].astype(int))

df_inverse.head()

In [ ]:
# Melakukan analisis deskriptif minimal mean, min dan max untuk fitur numerik dan mode untuk kategorikal seperti pada basic tetapi menggunakan data yang sudah diinverse.
agg_summary_num = df_inverse.groupby('Target')[numerical_cols].agg(['mean', 'min', 'max']).round(2).T

# Kelompokkan (groupby) 'df_inverse' berdasarkan 'Target' dan hitung agregasi untuk 'categorical_cols'.
#   - Hitung agregasi (agg) 'mode' (nilai yang paling sering muncul).
#   - (Kita gunakan 'lambda x: x.mode()[0]' untuk mengambil nilai mode pertama)
agg_summary_cat = df_inverse.groupby('Target')[categorical_cols].agg(lambda x: x.mode()[0]).round(2).T

display(agg_summary_num)
display(agg_summary_cat)

## Menjelaskan karakteristik tiap cluster berdasarkan rentangnya setelah **Inverse**.
1. **Cluster 0: Established & Engaged Users**:
  - **Rata-rata (mean) TransactionAmount:** <Fitur>: 255.55
  - **Rata-rata (mean) CustomerAge:** <Fitur>: 45.06
  - **Rata-rata (mean) TransactionDuration:** <Fitur>: 121.12
  - **Rata-rata (mean) LoginAttempts:** <Fitur>: 1
  - **Rata-rata (mean) AccountBalance:** <Fitur>: 5142.17
<<Setelah inverse>

  - **Analisis:** Cluster ini menggambarkan kelompok pengguna dengan usia yang relatif lebih matang, durasi transaksi yang lebih lama, serta saldo akun yang lebih tinggi. Hal ini menunjukkan pola penggunaan yang lebih stabil dan keterlibatan yang lebih tinggi dalam sistem. Pengguna dalam cluster ini cenderung memiliki aktivitas yang lebih konsisten, sehingga dapat dikategorikan sebagai pengguna yang lebih berpengalaman dan memiliki tingkat engagement yang lebih tinggi. Sehingga rekomendasi pada kelompok nasabah ini adalah dengan memberikan promosi, cashback, atau program loyalitas untuk meningkatkan keterlibatan, serta menawarkan produk yang sederhana dan mudah digunakan agar nasabah lebih aktif dalam bertransaksi.


2. **Cluster 1: Casual & Low Engagement Users**
  - **Rata-rata (mean) TransactionAmount:** <Fitur>: 258.15
  - **Rata-rata (mean) CustomerAge:** <Fitur>: 44.33
  - **Rata-rata (mean) TransactionDuration:** <Fitur>: 117.30
  - **Rata-rata (mean) LoginAttempts:** <Fitur>: 1
  - **Rata-rata (mean) AccountBalance:** <Fitur>: 5058.81

  - **Analisis:** Cluster ini menggambarkan kelompok pengguna dengan usia yang relatif sedikit lebih muda, durasi transaksi yang lebih singkat, serta saldo akun yang lebih rendah. Meskipun memiliki nilai transaksi yang sedikit lebih tinggi, pola penggunaan mereka cenderung tidak terlalu intens. Hal ini menunjukkan bahwa pengguna dalam cluster ini kemungkinan melakukan transaksi bernilai cukup besar namun dengan interaksi yang lebih sederhana dan tidak terlalu kompleks, sehingga dapat dikategorikan sebagai pengguna dengan tingkat keterlibatan yang relatif lebih rendah. Sehingga rekomendasi pada kelompok nasabah ini adalah dengan menawarkan produk premium seperti investasi, tabungan berjangka, atau layanan prioritas, karena mereka cenderung memiliki keterlibatan dan kemampuan finansial yang lebih baik.

In [ ]:
# Periksa kembali data yang telah di-inverse.
df_inverse.head()

In [ ]:
# Simpan Data Inverse
df_inverse.to_csv('data_clustering_inverse.csv', index=False)